## Decoder model 

In this section, we will discuss decoder models, with a focus on GPT3 as an example. We will cover the architecture, language modeling, and training process.

The basic architecture is extremely similar to the encoder model and comprises a series
of transformer layers that operate on learned word embeddings. However, the goal is
different. The encoder aimed to build a representation of the text that could be fine-
tuned to solve a variety of more specific NLP tasks. Conversely, the decoder has one
purpose: to generate the next token in a sequence. It can generate a coherent text
passage by feeding the extended sequence back into the model.

### Language modeling 

GPT3 is an autoregressive language model. This is easiest to understand with a concrete
example. Consider the sentence 

> It takes great courage to let yourself appear weak. 

For simplicity, let’s assume that the tokens are the full words. The probability of the full
sentence can be factored as:

\begin{eqnarray}
  Pr(\mbox{\textcolor{red}{It takes great courage to let yourself appear weak}}) &=&\nonumber\\
  &&\hspace{-8cm}Pr(\mbox{\textcolor{red}{It}})\times Pr(\mbox{\textcolor{red}{takes}}|\mbox{\textcolor{red}{It}})\times Pr(\mbox{\textcolor{red}{great}}|\mbox{\textcolor{red}{It takes}})\times Pr(\mbox{\textcolor{red}{courage}}|\mbox{\textcolor{red}{It takes great}})\times\nonumber \\
  &&\hspace{-8cm}Pr(\mbox{\textcolor{red}{to}}|\mbox{\textcolor{red}{It takes great courage}})\times Pr(\mbox{\textcolor{red}{let}}|\mbox{\textcolor{red}{It takes great courage to}})\times\nonumber\\
  &&\hspace{-8cm}Pr(\mbox{\textcolor{red}{yourself}}|\mbox{\textcolor{red}{It takes great courage to let}})\times\nonumber\\
  &&\hspace{-8cm}Pr(\mbox{\textcolor{red}{appear}}|\mbox{\textcolor{red}{It takes great courage to let yourself}})\times\nonumber\\
  &&\hspace{-8cm}Pr(\mbox{\textcolor{red}{weak}}|\mbox{\textcolor{red}{It takes great courage to let yourself appear}}).
 \end{eqnarray}

An autoregressive model predicts the conditional distributions $Pr(t_n|t_1,...,t_{n-1})$ of
each token given all the prior tokens, and hence indirectly computes the joint proba-
bility $Pr(t_1,t_2,...,t_N )$ of all $N$ tokens:

\begin{eqnarray}\label{eq:transformer_autoregressive}
  Pr(t_{1},t_{2},\ldots, t_{N}) = Pr(t_{1})\prod_{n=2}^{N}Pr(t_{n}|t_{1},\ldots, t_{n-1}).
 \end{eqnarray}

The autoregressive formulation demonstrates the connection between maximizing the
joint probability of the tokens and the next token prediction task.

### Masked self-attention 

To train a decoder, we seek parameters that maximize the log probability of the input
text under the autoregressive model (i.e., that maximize the sum of the log conditional
probability terms). Ideally, we would pass in the whole sentence and compute all the
log probabilities and gradients in the same forward pass rather than doing a forward
pass for each token in the sentence. However, if we pass in the full sentence, the term
computing log [Pr(great|It takes)] would have access to both the answer great and the
right context courage to let yourself appear weak. Hence, the system can cheat rather
than learn to predict the following words and won’t train properly.

Fortunately, the tokens only interact in the self-attention layers in a transformer
network. Hence, the problem can be resolved by ensuring that the attention to the
answer and the right context is zero. This can be achieved by setting the corresponding
dot products in the self-attention computation (equation 12.5) to negative infinity before
they are passed through the $\textbf{softmax[•]}$ function. This is known as masked self-attention.
The effect is to make the weight of all the upward-angled arrows in figure 12.1 zero.

The entire decoder network operates as follows. The input text is tokenized, and the
tokens are converted to embeddings. The embeddings are passed into the transformer
network, but now the transformer layers use masked self-attention so that they can
only attend to the current and previous tokens. Each of the output embeddings can be
thought of as representing a partial sentence, and for each, the goal is to predict the next
token in the sequence. Consequently, after the transformer layers, a single linear layer
maps each output embedding to the size of the vocabulary, followed by a $\textbf{softmax[•]}$
function that converts these values to probabilities. During training, we aim to maximize
the sum of the log probabilities of the next token in the ground truth sequence at every
position using a standard multiclass cross-entropy loss (figure 12.12).

### Generating text from a decoder 

The autoregressive language model is the first example of a generative model discussed
in this book. Since it defines a probability model over text sequences, it can be used
to sample new examples of plausible text. To generate from the model, we start with
an input sequence of text (which might be just the special <start> token indicating
the beginning of the sequence) and feed this into the network, which then outputs the
probabilities over possible subsequent tokens. We can then either pick the most likely
token or sample from this probability distribution. The new extended sequence can be
fed back into the decoder network to yield the probability distribution over the next token. By repeating this process, we can generate large bodies of text. The computation
can be made quite eﬀicient as prior embeddings do not depend on subsequent ones due
to the masked self-attention. Hence, much of the earlier computation can be recycled as
we generate subsequent tokens.

In practice, many strategies can make the output text more coherent. For example,
<mark>beam search</mark> keeps track of multiple possible sentence completions to find the overall most
likely sequence of words (which is not necessarily found by greedily choosing the most
likely word at each step). <mark>Top-k sampling</mark> randomly draws the next word from only the
top-K most likely possibilities to prevent the system from accidentally choosing from the
long tail of low-probability tokens and leading to an unnecessary linguistic dead end.

<img src="assets/Chap12/TransformerDecoder.svg" style="filter: invert(1);" width="100%">

Training GPT3-type decoder network. The tokens are mapped to
word embeddings with a special <start> token at the beginning of the sequence.
The embeddings are passed through a series of transformer layers that use masked
self-attention. Here, each position in the sentence can only attend to its own
embedding and those of tokens earlier in the sequence (orange connections). The
goal at each position is to maximize the probability of the following ground truth
token in the sequence. In other words, at position one, we want to maximize the
probability of the token It; at position two, we want to maximize the probability
of the token takes; and so on. Masked self-attention ensures the system cannot
cheat by looking at subsequent inputs. The autoregressive task has the advantage
of making efficient use of the data since every word contributes a term to the loss
function. However, it only exploits the left context of each word.

### GPT3 and few-shot learning 

Large language models like GPT3 apply these ideas on a massive scale. In GPT3, the
sequence lengths are 2048 tokens long, and the total batch size is 3.2 million tokens.
There are 96 transformer layers (some of which implement a sparse version of attention),
each processing a word embedding of size 12288. There are 96 heads in the self-attention layers, and the value, query, and key dimension is 128. It is trained with 300 billion
tokens and contains 175 billion parameters.

Here’s an example of completing text from the GPT3 model where the text provided
to the model is in orange, and the generated text is in cyan:

> Understanding Deep Learning is a new textbook from MIT Press by Simon Prince that<font color="cyan">’s de-
signed to offer an accessible, broad introduction to the field. Deep learning is a branch of
machine learning that is concerned with algorithms that learn from data that is unstructured
or unlabeled. The book is divided into four sections:
> 
> 1. Introduction to deep learning
> 
> 2. Deep learning architecture
> 
> 3. Deep learning algorithms
> 
> 4. Applications of deep learning
>  
> The first section offers an introduction to deep learning, including its history and origins.
The second section covers deep learning architecture, discussing various types of neural net-
works and their applications. The third section dives into deep learning algorithms, including
supervised and unsupervised learning, reinforcement learning, and more. The fourth section
applies deep learning to various domains, such as computer vision, natural language processing,
and robotics.</font>

This continuation is plausible, albeit only partially accurate.
One surprising property of learning models on this scale is that they can perform many
tasks without fine-tuning. If we provide several examples of correct question/answer pairs
and then another question, they often answer the final question correctly by completing
the sequence. One example is correcting English grammar:

> Poor English input: I eated the purple berries. \
> Good English output: I ate the purple berries. \
> Poor English input: Thank you for picking me as your designer. I’d appreciate it. \
> Good English output: Thank you for choosing me as your designer. I appreciate it. \
> Poor English input: The mentioned changes have done. or I did the alteration that you
requested. or I changed things you wanted and did the modifications. \
> Good English output: The requested changes have been made. or I made the alteration that
you requested. or I changed things you wanted and made the modifications. \
> Poor English input: I’d be more than happy to work with you in another project. \
> Good English output: <font color="cyan">I’d be more than happy to work with you on another project.</font> \
(result from Brown et al., 2020)

Here, the text containing the paired examples in orange was provided as context for
GPT3, and the system then generated the correct answer in <font color="cyan">cyan</font>. This phenomenon
extends to many situations, including generating code snippets based on natural language
descriptions, arithmetic, translating between languages, and answering questions about
text passages. Consequently, it is argued that enormous language models are <mark>few-shot
learners</mark>; they can learn to do novel tasks based on just a few examples. However,
performance is erratic in practice, and the extent to which it is extrapolating from
learned examples rather than merely interpolating or copying verbatim is unclear.

<img src="assets/Chap12/TransformerEncoderDecoder.svg" style="filter: invert(1);" width="100%">

Encoder-decoder architecture. Two sentences are passed to the
system with the goal of translating the first into the second. a) The first sentence
is passed through a standard encoder. b) The second sentence is passed through a
decoder that uses masked self-attention but also attends to the output embeddings
of the encoder using cross-attention (orange rectangle). The loss function is the
same as for the decoder model; we want to maximize the probability of the next
word in the output sequence.



In [ ]:
import re, collections
     

text = "a sailor went to sea sea sea "+\
                  "to see what he could see see see "+\
                  "but all that he could see see see "+\
                  "was the bottom of the deep blue sea sea sea"

Tokenize the input sentence To begin with the tokens are the individual letters and the whitespace token. So, we represent each word in terms of these tokens with spaces between the tokens to delineate them.

The tokenized text is stored in a structure that represents each word as tokens together with the count of how often that word occurs. We'll call this the vocabulary.

In [ ]:
def initialize_vocabulary(text):
  vocab = collections.defaultdict(int)
  words = text.strip().split()
  for word in words:
      vocab[' '.join(list(word)) + ' '] += 1
  return vocab
     

vocab = initialize_vocabulary(text)
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

Find all the tokens in the current vocabulary and their frequencies

In [ ]:
def get_tokens_and_frequencies(vocab):
  tokens = collections.defaultdict(int)
  for word, freq in vocab.items():
      word_tokens = word.split()
      for token in word_tokens:
          tokens[token] += freq
  return tokens
     

tokens = get_tokens_and_frequencies(vocab)
print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))

Find each pair of adjacent tokens in the vocabulary and count them. We will subsequently merge the most frequently occurring pair.

In [ ]:
def get_pairs_and_counts(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i],symbols[i+1]] += freq
    return pairs
     

pairs = get_pairs_and_counts(vocab)
print('Pairs: {}'.format(pairs))
print('Number of distinct pairs: {}'.format(len(pairs)))

most_frequent_pair = max(pairs, key=pairs.get)
print('Most frequent pair: {}'.format(most_frequent_pair))

Merge the instances of the best pair in the vocabulary

In [ ]:
def merge_pair_in_vocabulary(pair, vocab_in):
    vocab_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?

vocab = merge_pair_in_vocabulary(most_frequent_pair, vocab)
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

Update the tokens, which now include the best token 'se'

In [ ]:
tokens = get_tokens_and_frequencies(vocab)
print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))

Now let's write the full tokenization routine

In [ ]:
# TODO -- write this routine by filling in this missing parts,
# calling the above routines
def tokenize(text, num_merges):
  # Initialize the vocabulary from the input text
  # vocab = (your code here)

  for i in range(num_merges):
    # Find the tokens and how often they occur in the vocabulary
    # tokens = (your code here)

    # Find the pairs of adjacent tokens and their counts
    # pairs = (your code here)

    # Find the most frequent pair
    # most_frequent_pair = (your code here)
    print('Most frequent pair: {}'.format(most_frequent_pair))

    # Merge the code in the vocabulary
    # vocab = (your code here)

  # Find the tokens and how often they occur in the vocabulary one last time
  # tokens = (your code here)

  return tokens, vocab
     

tokens, vocab = tokenize(text, num_merges=22)
     

print('Tokens: {}'.format(tokens))
print('Number of tokens: {}'.format(len(tokens)))
print('Vocabulary: {}'.format(vocab))
print('Size of vocabulary: {}'.format(len(vocab)))

TODO - Consider the input text:

"How much wood could a woodchuck chuck if a woodchuck could chuck wood"

How many tokens will there be initially and what will they be? How many tokens will there be if we run the tokenization routine for the maximum number of iterations (merges)?

When you've made your predictions, run the code and see if you are correct.